In [1]:
# %pip install qiskit==1.2.4
# %pip install qiskit-aer==0.15.1
# %pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 with an attacker (Eve)

This notebook simulates BB84 in the presence of an eavesdropper, **Eve**, who
performs a *intercept-and-resend* attack: for every qubit Alice
sends, Eve measures it in a quantumly-chosen random basis and then re-prepares
a fresh qubit in the basis she used, carrying the bit she just measured.

Because Eve's basis disagrees with Alice's on roughly half the qubits, she
randomises the value Bob receives whenever bases disagree. After sifting (when
Alice's and Bob's bases agree), Eve's interference still corrupts about a
quarter of the bits, so the public sample is expected to show an error rate
near 25% — well above the 11% threshold — and the protocol raises an alarm.

In [2]:
N = 100              # number of qubits Alice transmits
THRESHOLD = 0.11     # eavesdropping detection threshold 

backend = BasicSimulator()

## Alice prepares
Alice generates her secret bits and her basis choices.

In [3]:
# Alice generates N random bits by measuring |+> once each.
alice_bits = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    alice_bits_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    alice_bits.append(int(alice_bits_result))

# Alice generates N random basis choices the same way.
alice_bases = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    alice_base_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    alice_bases.append(int(alice_base_result))

print("Alice's bits   :", "".join(str(b) for b in alice_bits))
print("Alice's bases  :", "".join(str(b) for b in alice_bases))
print("\nbasis bit 0 -> standard basis {|0>, |1>}\nbasis bit 1 -> diagonal basis {|+>, |->}")

Alice's bits   : 1101010000101101010011010000111011110010010110000001111100101011101000110001110010011010010000110101
Alice's bases  : 0110110001010111100011001011000111101100001101001111011111011000100101100111101001000100011100101101

basis bit 0 -> standard basis {|0>, |1>}
basis bit 1 -> diagonal basis {|+>, |->}


## Bob chooses bases
Bob picks a random measurement basis for every incoming qubit.

In [4]:
# Bob, independently and in advance, chooses a random basis for every incoming qubit.
bob_bases = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    bob_base_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    bob_bases.append(int(bob_base_result))

print("Bob's bases    :", "".join(str(b) for b in bob_bases))
print("\nbasis bit 0 -> standard basis {|0>, |1>}\nbasis bit 1 -> diagonal basis {|+>, |->}")

Bob's bases    : 0110000111010101000111101110000011111111011001011001000100001100000100001010100100000001100000000110

basis bit 0 -> standard basis {|0>, |1>}
basis bit 1 -> diagonal basis {|+>, |->}


## Eve intercepts
Eve generates her own random basis for every qubit.

In [5]:
# Eve generates her own random basis for every qubit
eve_bases = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    eve_base_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    eve_bases.append(int(eve_base_result))

print("Eve's bases    :", "".join(str(b) for b in eve_bases))
print("\nbasis bit 0 -> standard basis {|0>, |1>}\nbasis bit 1 -> diagonal basis {|+>, |->}")

Eve's bases    : 1111110111001011111011011111000100000110001010111101101100000110111101011001010110111011101010011100

basis bit 0 -> standard basis {|0>, |1>}
basis bit 1 -> diagonal basis {|+>, |->}


## Quantum transmission with Eve in the middle
For each qubit: (A) Alice prepares, Eve measures in her basis; (B) Eve re-prepares the qubit in her own basis with her measured value, Bob measures in his basis.

In [6]:
# Intercept-and-resend attack.
#
# For each transmission we use TWO circuits:
#   Circuit A: Alice prepares the qubit, Eve measures in eve_bases[i] -> eve_bits[i].
#              (Measurement collapses the qubit; the state Bob receives is what Eve re-prepares.)
#   Circuit B: Eve re-prepares (eve_bits[i], eve_bases[i]) and sends to Bob,
#              Bob measures in bob_bases[i] -> bob_bits[i].
eve_bits = []
bob_bits = []
for i in range(N):
    # --- Circuit A: Alice prepares, Eve measures ---
    qcA = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qcA.x(0)
    if alice_bases[i] == 1:
        qcA.h(0)
    # Eve's measurement basis
    if eve_bases[i] == 1:
        qcA.h(0)
    qcA.measure(0, 0)
    eve_bits_result = backend.run(transpile(qcA, backend), shots=1, memory=True).result().get_memory()[0]
    eve_bits.append(int(eve_bits_result))

    # --- Circuit B: Eve re-prepares in her basis, Bob measures ---
    qcB = QuantumCircuit(1, 1)
    if eve_bits[i] == 1:
        qcB.x(0)
    if eve_bases[i] == 1:
        qcB.h(0)
    # Bob's measurement basis
    if bob_bases[i] == 1:
        qcB.h(0)
    qcB.measure(0, 0)
    bob_bits_result = backend.run(transpile(qcB, backend), shots=1, memory=True).result().get_memory()[0]
    bob_bits.append(int(bob_bits_result))

print("Eve's bits     :", "".join(str(b) for b in eve_bits))
print("Bob's bits     :", "".join(str(b) for b in bob_bits))

Eve's bits     : 0101010100111001011011010100111010010000010111110011011100101101100000001011011100111110110010100100
Bob's bits     : 0100110100101001000111100101111010001001010101110011111100101101101000011010101110010110111000100100


## Basis reconciliation (sifting)
Alice and Bob keep only the positions where their bases matched.

In [7]:
# Alice and Bob publicly announce their bases and keep only the positions where they agree.
basis_match = [alice_bases[i] == bob_bases[i] for i in range(N)]
sifted_alice = [alice_bits[i] for i in range(N) if basis_match[i]]
sifted_bob = [bob_bits[i] for i in range(N) if basis_match[i]]

print("Basis match    :", "".join("1" if match else "0" for match in basis_match))
print("Sifted length  :", len(sifted_alice))
print("Sifted Alice   :", "".join(str(bits) for bits in sifted_alice))
print("Sifted Bob     :", "".join(str(bits) for bits in sifted_bob))

Basis match    : 1111001001111101011011011010111011101100101011101001100100101011011110010010110010111010000011010100
Sifted length  : 56
Sifted Alice   : 11010010111101110011111100001000111111101001011101110011
Sifted Bob     : 01000010101001100011110010000110111110101001110101010001


## Eavesdropping check
A random half of the sifted key is publicly compared. With Eve present the sample's mismatch rate should be near 25%.

In [8]:
# Eavesdropping check: for each position in the sifted key we randomly choose whether to sample it.
#   result == 1 -> the bit is publicly compared.
#   result == 0 -> the bit is kept as part of the final secret key.
sample_mask = []
for _ in range(len(sifted_alice)):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    mask_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    sample_mask.append(int(mask_result))

sample_indices = [i for i, mask in enumerate(sample_mask) if mask == 1]
sample_alice   = [sifted_alice[i] for i in sample_indices]
sample_bob     = [sifted_bob[i]   for i in sample_indices]

mismatch_count = sum(1 for a, b in zip(sample_alice, sample_bob) if a != b)
error_rate     = mismatch_count / len(sample_alice) if sample_alice else 0.0

print("Sample indices :", sample_indices)
print("Sample Alice   :", "".join(str(b) for b in sample_alice))
print("Sample Bob     :", "".join(str(b) for b in sample_bob))
print(f"Mismatches     : {mismatch_count} / {len(sample_alice)}")
print(f"Error rate     : {error_rate:.3f}  (threshold = {THRESHOLD})")

Sample indices : [1, 4, 5, 7, 8, 9, 12, 16, 18, 20, 21, 22, 23, 27, 28, 29, 34, 35, 38, 43, 44, 45, 46, 48, 52]
Sample Alice   : 1000110011111010111101100
Sample Bob     : 1000100011100001111111000
Mismatches     : 7 / 25
Error rate     : 0.280  (threshold = 0.11)


## Final key and verdict
If the error rate exceeds the threshold the protocol aborts.

In [9]:
final_key_alice = [sifted_alice[i] for i, mask in enumerate(sample_mask) if mask == 0]
final_key_bob   = [sifted_bob[i]   for i, mask in enumerate(sample_mask) if mask == 0]

print("Final key Alice:", "".join(str(bits) for bits in final_key_alice))
print("Final key Bob  :", "".join(str(bits) for bits in final_key_bob))
print("Keys match     :", final_key_alice == final_key_bob)
print("Final key len  :", len(final_key_alice))

if error_rate > THRESHOLD:
    print(f"\nALARM: error rate {error_rate:.3f} exceeds threshold {THRESHOLD}.")
    print("An eavesdropper is likely present -- ABORT and discard the key.")
else:
    print(f"\nOK: error rate {error_rate:.3f} is within threshold {THRESHOLD}.")
    print("No eavesdropping detected -- final key accepted.")

Final key Alice: 1011111110100000111101001111011
Final key Bob  : 0001101100110010111001001101001
Keys match     : False
Final key len  : 31

ALARM: error rate 0.280 exceeds threshold 0.11.
An eavesdropper is likely present -- ABORT and discard the key.
